# Freight Graph in Neo4j (Ch 11 Graph Databases)
โหลดกราฟขนส่งเข้า Neo4j แล้วตอบคำถามธุรกิจ 5 ข้อด้วย Cypher คำถามเหล่านี้เป็น **multi-hop** (ฐาน → จังหวัดในรัศมี → ปลายทาง → ฐาน) ซึ่งกราฟตอบตรงกว่าตาราง

รันหลัง `Freight_Capstone_TH.ipynb` (ใช้ `data/` และ `results/`) ต้องมี Neo4j Aura Free: ตั้ง env `NEO4J_CREDENTIALS_FILE` ชี้ไฟล์ credentials ที่ดาวน์โหลดจาก Aura (อยู่นอก repo ห้าม commit)

**Ontology (Ch 11)**
- `(:Province {name, group, out_trips, in_trips, lat, lng})` + label `:Hub` สำหรับฐาน A_3
- `(:Province)-[:SHIPS_TO {product, trips, tons}]->(:Province)` แยกตามสินค้า (multigraph, Ch 3)
- `(:Hub)-[:REACHES {km}]->(:Province)` ภายใน 150 กม.

| CQ | คำถาม | trace |
|---|---|---|
| 1 | เที่ยวที่ต้นทางและปลายทางอยู่ในรัศมีฐาน (งานขากลับ) แยกตามคู่ฐาน | Hub-REACHES→P-SHIPS_TO→P←REACHES-Hub |
| 2 | รถจากแต่ละฐานไปจบที่ไหนมากที่สุด (หางานขากลับที่ไหน) | Hub-REACHES→P-SHIPS_TO→P |
| 3 | เส้นทางกากน้ำตาล (ต้องใช้รถแทงก์) ที่รับได้ | Hub-REACHES→P-SHIPS_TO{กากน้ำตาล}→P |
| 4 | ต้นทางใหญ่ที่ยังไม่มีฐานเข้าถึง (ผู้สมัครฐานที่ 4) | P ที่ไม่มี REACHES |
| 5 | จังหวัดในรัศมีที่ส่งไปท่าเรือ (ชลบุรี, สมุทรปราการ) | Hub-REACHES→P-SHIPS_TO→Port |

In [1]:
import os, json
from pathlib import Path
import pandas as pd, numpy as np
from neo4j import GraphDatabase
try:  # use the OS trust store (keeps TLS verification on; fixes networks whose root CA is only in the OS keychain)
    import truststore; truststore.inject_into_ssl()
except ImportError: pass
cred=Path(os.environ.get('NEO4J_CREDENTIALS_FILE','')).expanduser()
env={k.strip():v.strip() for k,v in (l.split('=',1) for l in cred.read_text().splitlines() if '=' in l and not l.startswith('#'))} if cred.is_file() else os.environ
driver=GraphDatabase.driver(env['NEO4J_URI'],auth=(env['NEO4J_USERNAME'],env['NEO4J_PASSWORD']));driver.verify_connectivity()
q=lambda cy,**kw:pd.DataFrame([r.data() for r in driver.execute_query(cy,**kw).records])
print('connected')

connected


In [2]:
# Same scope as the main notebook
P=['ข้าวเปลือก','ข้าว','น้ำตาลทราย','กากน้ำตาล','อาหารสัตว์','ข้าวโพด','ผลิตภัณฑ์มันสำปะหลัง']
raw=pd.read_csv('data/otp_69_01_69.csv')
for c in ['product','province_origin','province_destination']:raw[c]=raw[c].astype('string').str.strip()
d=raw[raw['product'].isin(P)&raw.province_origin.ne(raw.province_destination)]
D=pd.read_csv('data/straight_line_distance_km.csv',index_col=0);coords=pd.read_csv('data/province_coordinates_used.csv').set_index('province')
cc=pd.read_csv('results/community_coverage.csv');cc=cc[cc.model=='A_3']
group={v.strip():int(r.community) for _,r in cc.iterrows() for v in r.members.split(' / ')}
hubs=pd.read_csv('results/models.csv').set_index('model').sites['A_3'].split(' / ')
ships=d.groupby(['province_origin','province_destination','product'],as_index=False).agg(trips=('trips','sum'),tons=('shipping','sum'))
prov=pd.DataFrame({'name':D.index}).assign(group=lambda x:x.name.map(group),
    out_trips=lambda x:x.name.map(d.groupby('province_origin').trips.sum()).fillna(0),in_trips=lambda x:x.name.map(d.groupby('province_destination').trips.sum()).fillna(0),
    lat=lambda x:x.name.map(coords.LAT),lng=lambda x:x.name.map(coords.LNG))
reach=[dict(hub=h,prov=v,km=float(D.at[h,v])) for h in hubs for v in D.index if D.at[h,v]<=150]
print(len(prov),'provinces',len(ships),'SHIPS_TO',len(reach),'REACHES')

77 provinces 4955 SHIPS_TO 41 REACHES


In [3]:
# Load (idempotent: wipe and reload)
driver.execute_query('MATCH (n) DETACH DELETE n')
driver.execute_query('CREATE CONSTRAINT province_name IF NOT EXISTS FOR (p:Province) REQUIRE p.name IS UNIQUE')
driver.execute_query('UNWIND $rows AS r MERGE (p:Province {name:r.name}) SET p += r',rows=prov.to_dict('records'))
driver.execute_query('UNWIND $h AS name MATCH (p:Province {name:name}) SET p:Hub',h=hubs)
driver.execute_query('''UNWIND $rows AS r MATCH (a:Province {name:r.province_origin}),(b:Province {name:r.province_destination})
CREATE (a)-[:SHIPS_TO {product:r.product,trips:r.trips,tons:r.tons}]->(b)''',rows=ships.to_dict('records'))
driver.execute_query('UNWIND $rows AS r MATCH (h:Hub {name:r.hub}),(p:Province {name:r.prov}) CREATE (h)-[:REACHES {km:r.km}]->(p)',rows=reach)
tot=q('MATCH ()-[s:SHIPS_TO]->() RETURN sum(s.trips) AS trips').trips[0]
assert abs(tot-d.trips.sum())<1e-6, 'Neo4j totals must equal pandas totals'
print('loaded; trips',tot)

loaded; trips 990030


In [4]:
# CQ1 Backhaul-ready lanes: origin and destination both within reach of a hub, by hub pair
cq1=q('''MATCH (h1:Hub)-[:REACHES]->(a:Province)-[s:SHIPS_TO]->(b:Province)<-[:REACHES]-(h2:Hub)
RETURN h1.name AS pickup_hub, h2.name AS delivery_hub, sum(s.trips) AS trips ORDER BY trips DESC''')
lane_total=q('''MATCH (a:Province)-[s:SHIPS_TO]->(b:Province) WHERE EXISTS {(:Hub)-[:REACHES]->(a)} AND EXISTS {(:Hub)-[:REACHES]->(b)}
RETURN sum(s.trips)*100.0/$t AS pct''',t=float(tot)).pct[0]
lane_nb=pd.read_csv('results/lane_coverage.csv').query("set=='A_3'").lane_coverage.iloc[0]
assert abs(lane_total-lane_nb)<1e-6, 'Cypher lane coverage must match notebook 11.7'
print(f'Lane coverage of A_3 = {lane_total:.2f}% (matches notebook)');cq1

Lane coverage of A_3 = 75.48% (matches notebook)


,pickup_hub,delivery_hub,trips
0,สระบุรี,สระบุรี,348267
1,พิษณุโลก,สระบุรี,123802
2,มหาสารคาม,สระบุรี,95676
3,พิษณุโลก,พิษณุโลก,74563
4,มหาสารคาม,มหาสารคาม,48547
5,สระบุรี,พิษณุโลก,30714
6,สระบุรี,มหาสารคาม,19381
7,พิษณุโลก,มหาสารคาม,5117
8,มหาสารคาม,พิษณุโลก,1236


In [5]:
# CQ2 Where do trucks from each hub's pickup area end up? (where to sell return loads)
q('''MATCH (h:Hub)-[:REACHES]->(a:Province)-[s:SHIPS_TO]->(b:Province)
WITH h, b, sum(s.trips) AS trips ORDER BY trips DESC
WITH h, collect({dest:b.name, trips:trips, hub_nearby:EXISTS {(:Hub)-[:REACHES]->(b)}})[..5] AS top
UNWIND top AS t RETURN h.name AS hub, t.dest AS destination, t.trips AS trips, t.hub_nearby AS return_load_near_our_hub''')

,hub,destination,trips,return_load_near_our_hub
0,สระบุรี,ชลบุรี,67342,True
1,สระบุรี,พระนครศรีอยุธยา,48599,True
2,สระบุรี,สมุทรปราการ,39197,True
3,สระบุรี,สุพรรณบุรี,24866,True
4,สระบุรี,ฉะเชิงเทรา,21016,True
5,มหาสารคาม,ชลบุรี,40576,True
6,มหาสารคาม,นครราชสีมา,21982,True
7,มหาสารคาม,ขอนแก่น,13136,True
8,มหาสารคาม,กาฬสินธุ์,6144,True
9,มหาสารคาม,กรุงเทพมหานคร,6074,True


In [6]:
# CQ3 Molasses lanes we can pick up (need tanker trucks)
q('''MATCH (h:Hub)-[:REACHES]->(a:Province)-[s:SHIPS_TO {product:'กากน้ำตาล'}]->(b:Province)
RETURN h.name AS hub, a.name AS origin, b.name AS destination, s.trips AS trips ORDER BY trips DESC LIMIT 10''')

,hub,origin,destination,trips
0,พิษณุโลก,นครสวรรค์,อ่างทอง,4661
1,สระบุรี,อุทัยธานี,สุพรรณบุรี,2803
2,มหาสารคาม,อุดรธานี,ขอนแก่น,2661
3,พิษณุโลก,กำแพงเพชร,กาญจนบุรี,2349
4,พิษณุโลก,กำแพงเพชร,สุพรรณบุรี,1790
5,สระบุรี,นครราชสีมา,ลพบุรี,1553
6,มหาสารคาม,กาฬสินธุ์,ชัยภูมิ,1061
7,สระบุรี,นครราชสีมา,ชลบุรี,1040
8,มหาสารคาม,หนองบัวลำภู,ชัยภูมิ,932
9,มหาสารคาม,สุรินทร์,ขอนแก่น,882


In [7]:
# CQ4 Largest origins no hub reaches: candidates for a 4th hub
q('''MATCH (p:Province) WHERE NOT EXISTS {(:Hub)-[:REACHES]->(p)}
RETURN p.name AS province, p.group AS community, p.out_trips AS origin_trips ORDER BY origin_trips DESC LIMIT 10''')

,province,community,origin_trips
0,อุบลราชธานี,0,35318.0
1,กาญจนบุรี,3,25329.0
2,เลย,0,20275.0
3,ราชบุรี,3,17378.0
4,เพชรบุรี,3,12648.0
5,ศรีสะเกษ,0,8890.0
6,สงขลา,3,6948.0
7,มุกดาหาร,0,6911.0
8,ลำปาง,2,4787.0
9,เชียงราย,2,4665.0


In [8]:
# CQ5 Reached provinces shipping to the port provinces (export lanes for sales targeting)
q('''MATCH (h:Hub)-[:REACHES]->(a:Province)-[s:SHIPS_TO]->(port:Province) WHERE port.name IN ['ชลบุรี','สมุทรปราการ']
RETURN h.name AS hub, a.name AS origin, port.name AS port, sum(s.trips) AS trips, collect(DISTINCT s.product) AS products ORDER BY trips DESC LIMIT 10''')

,hub,origin,port,trips,products
0,สระบุรี,นครราชสีมา,ชลบุรี,22226,"[กากน้ำตาล, ข้าว, ข้าวโพด, น้ำตาลทราย, ผลิตภัณ..."
1,สระบุรี,สระแก้ว,ชลบุรี,13366,"[กากน้ำตาล, ข้าว, ข้าวเปลือก, ข้าวโพด, น้ำตาลท..."
2,มหาสารคาม,สุรินทร์,ชลบุรี,9652,"[กากน้ำตาล, ข้าว, ข้าวเปลือก, น้ำตาลทราย, ผลิต..."
3,สระบุรี,ฉะเชิงเทรา,ชลบุรี,8442,"[ข้าว, ข้าวเปลือก, ผลิตภัณฑ์มันสำปะหลัง, อาหาร..."
4,มหาสารคาม,ชัยภูมิ,ชลบุรี,7754,"[กากน้ำตาล, ข้าว, น้ำตาลทราย, ผลิตภัณฑ์มันสำปะ..."
5,มหาสารคาม,ขอนแก่น,ชลบุรี,6076,"[กากน้ำตาล, ข้าว, น้ำตาลทราย, ผลิตภัณฑ์มันสำปะ..."
6,สระบุรี,สมุทรสาคร,สมุทรปราการ,6059,"[ข้าว, น้ำตาลทราย, อาหารสัตว์]"
7,พิษณุโลก,กำแพงเพชร,ชลบุรี,5410,"[กากน้ำตาล, ข้าว, ข้าวเปลือก, น้ำตาลทราย, ผลิต..."
8,มหาสารคาม,กาฬสินธุ์,ชลบุรี,5223,"[กากน้ำตาล, ข้าว, น้ำตาลทราย, ผลิตภัณฑ์มันสำปะ..."
9,สระบุรี,นครปฐม,สมุทรปราการ,4946,"[ข้าว, ข้าวเปลือก, อาหารสัตว์]"


### AI vs human (Ch 11–12)
- AI ช่วยร่าง ontology และ Cypher; ทีมตรวจทุก query ด้วย assert เทียบยอด pandas และ lane coverage จาก notebook หลัก (text-to-Cypher ไม่ถูกใช้โดยไม่ตรวจ)
- ใช้ graph DB เพราะคำถาม multi-hop (CQ1, CQ2, CQ5) เป็นงานประจำของฝ่ายขาย/จัดรถ ไม่ใช้ GraphRAG เพราะไม่มีเอกสารข้อความให้ ground (Ch 12: "use it when the question needs it")

In [9]:
driver.close()